### Tools

they extend what agents can do, letting them fetch data, code, db, do some actions 

tools are fuctions that get passed to chat models

Tools are declared by @tool and it contains

A schema, including the name of the tool, a description (functions doc string), and/or argument definitions (often a JSON schema)
A function or coroutine to execute.

In [1]:
import os
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3-32b")
response = model.invoke("why is tomato a fruit")
response

AIMessage(content='<think>\nOkay, so I need to figure out why tomatoes are considered fruits. Let me start by recalling what I know about fruits and vegetables. From what I\'ve been taught, fruits are the part of a plant that develops from the flower and contains seeds. Vegetables are usually other parts like roots, stems, or leaves. But I also remember that in everyday language, people often call tomatoes vegetables, even though botanically they might be different.\n\nSo, first, I should check the botanical definition of a fruit. A fruit, in botanical terms, is the mature ovary of a flowering plant, usually containing seeds. So if a tomato comes from the ovary of a flower and has seeds, then it would be a fruit. Let me confirm that. Tomatoes do have seeds inside them, and they develop from the flower\'s ovary. So that fits the botanical definition.\n\nBut then why is there confusion? I think the confusion comes from culinary uses. In cooking, fruits are usually sweet, like apples or b

In [2]:
from langchain.tools import tool

@tool
def get_weather(location: str):
    """ Get the weather at a location"""

    return f"It's sunny in {location}"

model_with_tools = model.bind_tools([get_weather])

In [3]:
response = model_with_tools.invoke("What's the weather like in Boston?")

print(response)

for tool_call in response.tool_calls:

    print(f"Tool : {tool_call["name"]}") #can be comtomized by @tool("web_search")
    print(f"Args: {tool_call["args"]}")

content='' additional_kwargs={'reasoning_content': 'Okay, the user is asking about the weather in Boston. I need to use the get_weather function. Let me check the function\'s parameters. It requires a location, which in this case is Boston. So I\'ll call the function with "Boston" as the location. I should make sure the arguments are correctly formatted in JSON. Let me structure the tool call accordingly.\n', 'tool_calls': [{'id': 'dh9akgr22', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 99, 'prompt_tokens': 154, 'total_tokens': 253, 'completion_time': 0.158961249, 'completion_tokens_details': {'reasoning_tokens': 75}, 'prompt_time': 0.007753219, 'prompt_tokens_details': None, 'queue_time': 0.023513931, 'total_time': 0.166714468}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_d58dbe76cd', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'm

description can be ustomised as  @tool("calculator", description="Performs arithmetic calculations. Use this for any math problems.")
 it iverrides teh given description

### Tool Execution

tool can return a string The return value is converted to a ToolMessage., command, object The object is serialized and sent back as tool output.
Like string returns, this does not directly update graph state. the model sees the fields and reason over them.


In [10]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

print("msg", messages) 

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    print("tc", tool_call)
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)
    print("msg1", messages) 


# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response)
print(final_response.content)
# "The current weather in Boston is 72°F and sunny."

msg [{'role': 'user', 'content': "What's the weather in Boston?"}, AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for the weather in Boston. I need to use the get_weather function. The function requires a location parameter. Boston is the location here. So I should call get_weather with location set to "Boston". Let me make sure there are no other parameters needed. The required field is just location, so that\'s all I need. Alright, the tool call should be correct.\n', 'tool_calls': [{'id': 'wf5gxjhn7', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 105, 'prompt_tokens': 153, 'total_tokens': 258, 'completion_time': 0.157571639, 'completion_tokens_details': {'reasoning_tokens': 81}, 'prompt_time': 0.007127157, 'prompt_tokens_details': None, 'queue_time': 0.042147122, 'total_time': 0.164698796}, 'model_name': 'qwen/qwen3-32b', 'system_finger